In [1]:
import os
import io
import shutil
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.preprocessing import image

from config.image_data_set_loader import ImageDataSetLoader

initializer = ImageDataSetLoader()
data_set = initializer.init_data_set()

print(f"TensorFlow version: {tf.__version__}")

I0000 00:00:1788917414.194561    2363 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788917414.195965    2363 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788917414.261654    2363 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788917415.690294    2363 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

Found 584744 images
TensorFlow version: 2.21.0


In [2]:
TRAIN_DIR = data_set.filter(
    lambda x: x["split"] == "Training"
)
VAL_DIR = data_set.filter(
    lambda x: x["split"] == "Validation"
)
BATCH_SIZE = 8
IMG_SIZE = (224, 224)  # VGG16 strictly demands 224x224 input arrays
EPOCHS = 42

print("\n--- Loading Datasets ---")
train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

val_ds = image_dataset_from_directory(
    VAL_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

class_names = sorted(
    set(TRAIN_DIR["label"])
)
num_classes = len(class_names)
print(f"Target Labels parsed from directory structure: {class_names}")


Filter:   0%|          | 2000/584744 [10:54<52:59:47,  3.05 examples/s]


KeyboardInterrupt: 

In [ ]:
print("\n--- Building VGG16 Architecture ---")
# 1. Download base weights without default 1000-class head
vgg16_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Freeze base layers to keep general features (edges, color boundaries) locked
vgg16_base.trainable = False

# 3. Build sequential container adding custom classification head
model = models.Sequential([
    # VGG16 expects image data formatted explicitly by its own library functions
    layers.Lambda(preprocess_input, input_shape=(224, 224, 3)),
    vgg16_base,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),  # Mitigates overfitting behavior
    layers.Dense(num_classes, activation='softmax')  # Yields prediction probabilities
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), # Sharpens class boundaries
    metrics=['accuracy']
)

model.summary()

In [ ]:
print("\n--- Initiating Model Fit Loop ---")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# Render accuracy and loss analytics side-by-side
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', marker='o')
plt.plot(epochs_range, val_acc, label='Validation Accuracy', marker='x')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', marker='o')
plt.plot(epochs_range, val_loss, label='Validation Loss', marker='x')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.grid(True)
plt.show()

# Save final weights to disk
model.save("../ML_models/fruit_veg_vgg16_model.keras")
print("Saved baseline model artifact to disk as 'fruit_veg_vgg16_model.keras'")

In [ ]:
def predict_single_image(img_path: str):
    """Utility function to load, print, and identify a single photo."""
    print(f"\n--- Evaluation Request for: {img_path} ---")
    if not os.path.exists(img_path):
        print(f"Error: Target path {img_path} does not exist.")
        return
        
    # Read image payload from file path
    img = image.load_img(img_path, target_size=IMG_SIZE)
    
    # In-line graph display inside Jupyter window
    plt.imshow(img)
    plt.axis('off')
    plt.show()
    
    # Transform into numerical matrix and construct batch dimension [1, 224, 224, 3]
    img_array = image.img_to_array(img)
    img_batch = np.expand_dims(img_array, axis=0)
    
    # Execute matrix mathematics
    predictions = model.predict(img_batch)[0]
    best_match_idx = np.argmax(predictions)
    confidence_score = predictions[best_match_idx]
    
    print(f"RESULT: Identified as '{class_names[best_match_idx]}' with {100 * confidence_score:.2f}% confidence.")

    for idx,class_name in enumerate(class_names):
        print(f"{class_name}: {predictions[idx] * 100:.2f}%")

# Test the prediction logic immediately using an image from our validation set
sample_test_path = "./banana.jpg"
predict_single_image(sample_test_path)